In [ ]:
%cd ..
%load_ext autoreload
%autoreload 2

# Configure logger to ignore everything to avoid cluttering the output
import logging
logging.getLogger().setLevel(logging.WARNING)

import dotenv # load env vars from .env
dotenv.load_dotenv()

from openai import OpenAI
import dotenv  
import os   

dotenv.load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

: 

# Direct Preference Optimization

#### Collect preferred and non-preferred response

In [4]:
%%capture
import os
import json
from src.examples.agent.design_w_promoter_vars import get_runner

DPO_OUTPUT_DIR = "datasets/design_w_promoter_vars_dataset_dpo"

start_index = 0
num_runs = 5
all_dpo_pairs = []

os.environ["OPENAI_MODEL"] = "gpt-4.1-2025-04-14"  

run_name = "run" 
for run_number in range(start_index, start_index + num_runs):
    run_id = f"{run_name}_{run_number}"

    runner = get_runner(max_rounds=25, max_attempts=3)

    print(f"Running {run_id}")
    dpo_pairs = runner.run_generate_preference_pair_on_tool_failures()
    print(f"Collected {len(dpo_pairs)} dpo pairs")
    
    all_dpo_pairs += dpo_pairs
    os.makedirs(f"{DPO_OUTPUT_DIR}/{runner.model}/{run_id}", exist_ok=True)
    with open(f"{DPO_OUTPUT_DIR}/{runner.model}/{run_id}/dpo_pairs.jsonl", "w") as f:
        for dpo_pair in dpo_pairs:
            f.write(json.dumps(dpo_pair) + "\n")
            
with open(f'{DPO_OUTPUT_DIR}/{runner.model}/{run_name}_all_dpo_pairs.jsonl', "w") as f:
    for dpo_pair in all_dpo_pairs:
        f.write(json.dumps(dpo_pair) + "\n")


 /----------------------------------------------------------------------------\
 |  yosys -- Yosys Open SYnthesis Suite                                       |
 |  Copyright (C) 2012 - 2025  Claire Xenia Wolf <claire@yosyshq.com>         |
 |  Distributed under an ISC-like license, type "license" to see terms        |
 \----------------------------------------------------------------------------/
 Yosys 0.50 (git sha1 b5170e1394f602c607e75bdbb1a2b637118f2086, clang++ 16.0.0 -fPIC -O3)

-- Running command `read_verilog /Users/admin/repos/geneforge/outputs/artifacts/20250714_111540/20250714_111622/NOR_gate_custom_promoters/verilogs/main.v; splitnets; hierarchy -auto-top; flatten; proc; opt -full; memory; opt -full; fsm; opt -full; techmap; opt -full; abc -g NOR; splitnets -ports; opt -full; opt_clean; clean -purge; flatten; show -format pdf -prefix /Users/admin/repos/geneforge/outputs/artifacts/20250714_111540/20250714_111622/NOR_gate_custom_promoters/output/main.v/main.v_ucf._yosys; wr

### Upload the training file and run the fine-tuning job

In [ ]:
from sklearn.model_selection import train_test_split

dpo_all_dpo_pairs_path = f"{DPO_OUTPUT_DIR}/{runner.model}/{run_name}_all_dpo_pairs.jsonl"
all_samples = [json.loads(line) for line in open(dpo_all_dpo_pairs_path)]

# split into train and test set
train_set_path = dpo_all_dpo_pairs_path.split("_all_dpo_pairs.jsonl")[0] + "_train.jsonl"
test_set_path = dpo_all_dpo_pairs_path.split("_all_dpo_pairs.jsonl")[0] + "_test.jsonl"
train_samples, test_samples = train_test_split(all_samples, test_size=0.2, random_state=42)

json.dump(train_samples, open(train_set_path, "w"))
json.dump(test_samples, open(test_set_path, "w"))
    
uploaded_file = client.files.create(
    file=open(train_set_path, "rb"),
    purpose="fine-tune",
)
uploaded_file.id

job = client.fine_tuning.jobs.create(
    training_file=uploaded_file.id,
    model="gpt-4.1-2025-04-14",
    method={
        "type": "dpo",
        "dpo": {
            "hyperparameters": {"beta": 0.1},
        },
    },
)

print(job)

In [ ]:
# Evaluate the original model. We can use the pregenerated chat histories. (TODO Split into test and train set)
from src.examples.agent.design_w_promoter_vars import score_run_from_directory

i = 0
scores_original = {}
for i in range(10):
    MODEL_NAME = "gpt-4.1-2025-04-14" # chat histories are stored under a model specific directory
    directory = f"outputs/chat_histories/{MODEL_NAME}/design_w_promoter_vars_dataset_{i}"
    score = score_run_from_directory(directory)
    scores_original[directory] = score

In [ ]:
# Run sessions with trained model (n=10)
TRAINED_MODEL_NAME_ID = "ft:gpt-4.1-2025-04-14:geneforge::BsWZyydP"

os.environ["OPENAI_MODEL"] = TRAINED_MODEL_NAME_ID

runner = get_runner(max_rounds=25, max_attempts=3)
runner.setup()
runner.generate_chat_histories(output_dir="outputs/chat_histories", base_run_name="design_w_promoter_vars_dataset", num_runs=10, start_index=0)

scores_ft = scores_for_runs_from_directory(f"outputs/chat_histories/{MODEL_NAME}")

# Reinforcement Learning 

#### Define the grader

In [ ]:
# https://platform.openai.com/docs/guides/reinforcement-fine-tuning
# The python source code must contain a grade function that takes in exactly two arguments and returns a float value as a grade.
# The first argument supplied to the grading function will be a dictionary populated with the model’s output during training for you to grade. output_json will only be populated if the output uses response_format.
# {
#     "choices": [...],
#     "output_text": "...",
#     "output_json": {},
#     "output_tools": [...]
# }
# The second argument supplied is a dictionary populated with input grading context. For evals, this will include keys from the data source. For fine-tuning this will include keys from each training data row.
# {
#     "reference_answer": "...",
#     "my_key": {...}
# }

import os
import requests
import importlib
import inspect

api_key = os.getenv("OPENAI_API_KEY")
headers = {"Authorization": f"Bearer {api_key}"}

grader_module = importlib.import_module("src.rl.graders.grade_design_w_promoters")
grader = {
    "type": "python",
    "source": inspect.getsource(grader_module)
}

response = requests.post(
    "https://api.openai.com/v1/fine_tuning/alpha/graders/validate",
    json={"grader": grader},
    headers=headers
)
print("validate request_id:", response.headers["x-request-id"])
print("validate response:", response.text)

#### Generate chat histories

In [ ]:
# Change the OPENAI_MODEL to the fine-tuned model
os.environ["OPENAI_MODEL"] = "gpt-o3-mini"

runner.setup()
print(runner.model)
runner.generate_chat_histories(output_dir="outputs/chat_histories", num_runs=10, start_index=0)

#### Validate grader

In [ ]:
import json
import requests

with open("outputs/chat_histories/gpt-o3-mini/design_w_promoter_vars_dataset_0/chat_history.json", "r") as f:
    chat_history = [json.loads(line) for line in f]
with open('outputs/chat_histories/gpt-o3-mini/design_w_promoter_vars_dataset_0/session_state.json', 'r') as f:
    session_state_history = [json.loads(line) for line in f]

print(chat_history[-1][0].keys())
print(session_state_history[-1]['history'][-1].keys())

In [ ]:
from src.rl.graders.grade_design_w_promoters import grade
sample = chat_history[-1][-1]
item = session_state_history[-1]['history'][-1]
grade({"choices": [sample]}, item)

In [ ]:
payload = {
  "grader": grader,
  "item": session_state_history[-1]['history'][-1],
  "model_sample": chat_history[-1][-1]
}

response = requests.post(
    "https://api.openai.com/v1/fine_tuning/alpha/graders/run",
    json=payload,
    headers=headers
)
print("run request_id:", response.headers["x-request-id"])
print("run response:", response.text)

In [ ]:
from src.examples.agent.design_w_promoter_vars import scores_for_runs_from_directory
scores = scores_for_runs_from_directory("outputs/chat_histories/gpt-o3-mini")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
scores_df = pd.DataFrame(scores).T
scores_df.head()

metrics=['num_messages', 
        'num_tool_calls', 
        'num_agent_messages', 
        'has_3_unique_promoters', 
        'has_3_new_promoters', 
        'has_3_new_promoter_sequences', 
        'has_correct_order', 
        'has_correct_truth_table']

plt.figure(figsize=(16, 6))
i = 1
for metric in metrics:
    plt.subplot(3, 3, i)
    if scores_df[metric].dtype == 'bool':
        plt.hist(scores_df[metric].astype(int), bins=2)
    else:
        plt.hist(scores_df[metric], bins=20)
    plt.title(metric)
    i += 1

plt.tight_layout()
plt.show()

### OpenPipe/ART
OpenAI reinforcement learning only supports single-turn RL.
So, to true multi-turn RL, we use https://github.com/OpenPipe/ART

### Verifiers
OpenAI reinforcement learning only supports single-turn RL.
So, to for multi-turn RL, let's try https://github.com/willccbb/verifiers/tree/main

In [ ]:
!pip install verifiers
!pip install tf-keras
!pip install vllm

In [ ]:
import json, copy
import verifiers as vf # pip install verifiers
from verifiers.envs.multiturn_env import MultiTurnEnv
from src.examples.agent.design_w_promoter_vars import (
    score_run, PROMPT, SYSTEM_PROMPT, DesignWithPromoterVarsRunner)

class DesignWithPromoterVarsEnv(MultiTurnEnv):
    """GRPO environment that runs one full WorkflowRunner episode."""
    def __init__(self, dataset, max_turns=25):
        super().__init__(dataset=dataset,
                         system_prompt=SYSTEM_PROMPT,
                         parser=None, # we don’t need XML parsing
                         rubric=None, # we’ll supply our own reward later
                         max_turns=max_turns)
        
    def is_completed(self, messages, state, kw):
        # stop when the workflow runner thinks we have cello_results
        return state.get("done", False)

    def env_response(self, messages, state, **kw):
        # messages[-1] is the assistant turn we just received
        # We execute any tool calls using the existing WorkflowRunner plumbing
        if "tool_calls" in messages[-1]:
            runner: DesignWithPromoterVarsRunner = state["runner"]
            last_asst = messages[-1]
            for tc in last_asst["tool_calls"]:
                fn = runner.tool_integration.call_tool_function
                out = fn(tc["function"]["name"],
                         json.loads(tc["function"]["arguments"]))
                messages.append({"role": "tool",
                                 "content": json.dumps(out),
                                 "tool_call_id": tc["id"]})
        # Use runner.check_success() as episode-done marker
        state["done"] = state["runner"].check_success()
        return {"role": "user", "content": ""}, state

    def reset(self, idx):
        prompt = self.dataset[idx]["prompt"]
        runner = DesignWithPromoterVarsRunner(
            example_name="DesignWithPromoterVarsRunner", prompt=prompt,
            system_prompt=SYSTEM_PROMPT,
            max_rounds=25, max_attempts=3)
        runner.setup() # gives us .messages, .session_state, etc.
        return (copy.deepcopy(runner.messages), # initial chat history
                {"runner": runner, "done": False}) # initial env state

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list([{"prompt": PROMPT}] * 10)

In [ ]:
import verifiers as vf
from verifiers.tools import python
import torch

vf_env = DesignWithPromoterVarsEnv(
    dataset=dataset,
    max_turns=25
)

# Training with tool environment
model_name = "Qwen/Qwen3-0.6B"
model, tokenizer = vf.get_model_and_tokenizer(model_name, model_kwargs=dict(use_cache=True))
run_name = "design_w_promoter_vars_grpo_" + model_name.split("/")[-1].lower()

In [ ]:

from verifiers.trainers.grpo_config import GRPOConfig

run_name = "design_w_promoter_vars_grpo_qwen3-0.6b"
training_args = GRPOConfig(
        output_dir=f"outputs/{run_name}",
        run_name=run_name,
        learning_rate=1e-6,
        lr_scheduler_type="constant_with_warmup",
        warmup_steps=10,
        num_train_epochs=1,
        max_steps=500,
        bf16=False,
        max_grad_norm=0.001,
        num_iterations=1,
        # max_concurrent=0,             # synchronise generation
        # num_batches_ahead=0,
        max_prompt_length=1024,
        max_completion_length=2048,
        per_device_train_batch_size=2,
        num_generations=8,
        gradient_accumulation_steps=4,
        gradient_checkpointing=False,
        save_strategy="steps",
        save_steps=500,
        save_only_model=True,
        logging_steps=1,
        log_on_each_node=False,
        log_completions=True,
        report_to=None,
        do_train=True
    )
training_args.num_iterations = 1
training_args.per_device_train_batch_size = 1
training_args.num_generations = 8

In [ ]:
trl vllm-serve \
    --model meta-llama/Llama-3.2-1B-Instruct \
    --dtype bfloat16 \
    --tensor-parallel-size 1 \
    --port 8000

In [ ]:
trainer = vf.GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    env=vf_env,
    args=training_args,
)
trainer.train()